In [2]:
#channel_benchmarking
import pandas as pd
import numpy as np
from scipy import stats
import json

np.random.seed(42)
df = pd.read_csv("data/global_ads_performance_dataset.csv")

groups = [df[df['platform']==p]['ROAS'].values for p in df['platform'].unique()]
h_stat, p_value = stats.kruskal(*groups)
n = len(df)
k = df['platform'].nunique()
epsilon_sq = (h_stat - k + 1) / (n - k)  

print("=== KRUSKAL-WALLIS: ROAS ~ platform ===")
print(f"H-statistic: {h_stat:.2f}, p-value: {p_value:.6f}")
print(f"Epsilon-squared (effect size): {epsilon_sq:.4f}")
print("(p<0.05 => khác biệt có ý nghĩa thống kê; epsilon^2: 0.01 nhỏ, 0.08 vừa, 0.26+ lớn)")

platforms = df['platform'].unique()
pairs = [(platforms[i], platforms[j]) for i in range(len(platforms)) for j in range(i+1, len(platforms))]
n_comparisons = len(pairs)

print("\n=== PAIRWISE MANN-WHITNEY U (Bonferroni-adjusted) ===")
pairwise_results = []
for p1, p2 in pairs:
    g1 = df[df['platform']==p1]['ROAS']
    g2 = df[df['platform']==p2]['ROAS']
    u_stat, p_raw = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    p_adj = min(p_raw * n_comparisons, 1.0)
    sig = "***" if p_adj < 0.001 else "**" if p_adj < 0.01 else "*" if p_adj < 0.05 else "ns"
    print(f"{p1} vs {p2}: U={u_stat:.0f}, p_raw={p_raw:.2e}, p_adj={p_adj:.2e} [{sig}]")
    pairwise_results.append({"pair": f"{p1} vs {p2}", "p_adj": p_adj, "significant": p_adj < 0.05})

print("\n=== BOOTSTRAP 95% CI: BLENDED ROAS per platform (10,000 resamples) ===")
bootstrap_results = {}
for p in platforms:
    sub = df[df['platform']==p]
    boot_roas = []
    for _ in range(10000):
        sample = sub.sample(n=len(sub), replace=True)
        boot_roas.append(sample['revenue'].sum() / sample['ad_spend'].sum())
    ci_low, ci_high = np.percentile(boot_roas, [2.5, 97.5])
    point = sub['revenue'].sum() / sub['ad_spend'].sum()
    print(f"{p}: Blended ROAS = {point:.2f}  [95% CI: {ci_low:.2f} - {ci_high:.2f}]")
    bootstrap_results[p] = {"point": round(point,2), "ci_low": round(ci_low,2), "ci_high": round(ci_high,2)}


print("\n=== KRUSKAL-WALLIS: ROAS ~ campaign_type, WITHIN mỗi platform ===")
within_platform_results = {}
for p in platforms:
    sub = df[df['platform']==p]
    g = [sub[sub['campaign_type']==ct]['ROAS'].values for ct in sub['campaign_type'].unique()]
    h, pv = stats.kruskal(*g)
    print(f"{p}: H={h:.2f}, p={pv:.4f} {'=> có ý nghĩa' if pv<0.05 else '=> không có ý nghĩa'}")
    within_platform_results[p] = {"H": round(h,2), "p_value": round(pv,4)}

import os
os.makedirs("dashboard_data", exist_ok=True)

def to_native(x):
    """Ép numpy types (bool_, float64, int64...) về kiểu Python thuần"""
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    return x

pairwise_results_clean = [
    {k: to_native(v) for k, v in item.items()} for item in pairwise_results
]

bootstrap_results_clean = {
    p: {k: to_native(v) for k, v in d.items()} for p, d in bootstrap_results.items()
}

within_platform_results_clean = {
    p: {k: to_native(v) for k, v in d.items()} for p, d in within_platform_results.items()
}

output = {
    "kruskal_wallis_platform": {
        "H": to_native(round(h_stat,2)),
        "p_value": to_native(p_value),
        "epsilon_sq": to_native(round(epsilon_sq,4))
    },
    "pairwise_platform": pairwise_results_clean,
    "bootstrap_blended_roas": bootstrap_results_clean,
    "within_platform_campaign_type": within_platform_results_clean
}

with open("dashboard_data/step2_channel_benchmarking.json", "w") as f:
    json.dump(output, f, indent=2)
print("\n✅ Saved: dashboard_data/step2_channel_benchmarking.json")

=== KRUSKAL-WALLIS: ROAS ~ platform ===
H-statistic: 209.05, p-value: 0.000000
Epsilon-squared (effect size): 0.1152
(p<0.05 => khác biệt có ý nghĩa thống kê; epsilon^2: 0.01 nhỏ, 0.08 vừa, 0.26+ lớn)

=== PAIRWISE MANN-WHITNEY U (Bonferroni-adjusted) ===
Google Ads vs TikTok Ads: U=84574, p_raw=3.88e-43, p_adj=1.16e-42 [***]
Google Ads vs Meta Ads: U=159084, p_raw=2.65e-21, p_adj=7.94e-21 [***]
TikTok Ads vs Meta Ads: U=169792, p_raw=2.88e-08, p_adj=8.63e-08 [***]

=== BOOTSTRAP 95% CI: BLENDED ROAS per platform (10,000 resamples) ===
Google Ads: Blended ROAS = 3.47  [95% CI: 3.24 - 3.70]
TikTok Ads: Blended ROAS = 7.62  [95% CI: 6.98 - 8.31]
Meta Ads: Blended ROAS = 5.66  [95% CI: 5.25 - 6.10]

=== KRUSKAL-WALLIS: ROAS ~ campaign_type, WITHIN mỗi platform ===
Google Ads: H=7.61, p=0.0548 => không có ý nghĩa
TikTok Ads: H=7.10, p=0.0689 => không có ý nghĩa
Meta Ads: H=1.44, p=0.6957 => không có ý nghĩa

✅ Saved: dashboard_data/step2_channel_benchmarking.json
